# 04 - Vegetation indices within the daily flux footprint (FFP climatology)

Instead of a fixed polygon, this notebook uses the actual source area of the tower on each HLS
overpass date. For every half hour of that day, the two-dimensional FFP model of
Kljun et al. (2015) is run with the tower's turbulence data (prepared in notebook 01). The half-hourly
footprints are summed, and the contour enclosing 90% of the daily footprint becomes the area over
which the spectral indices are averaged.

Requires `calc_footprint_FFP_climatology.py` from https://footprint.kljun.net in `../src/`
(see `src/README.md`).

In [ ]:
import sys
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
from pyproj import Transformer
from rasterio import mask
from shapely.geometry import Polygon

sys.path.append("../src")
from calc_footprint_FFP_climatology import FFP_climatology

In [ ]:
SITE = "ASP"          # "ASP" or "BAU"
SENSOR = "Landsat"    # "Landsat" or "Sentinel"

TOWERS = {            # lat, lon of the EC towers
    "ASP": (40.1585, -103.132),
    "BAU": (40.1503, -103.145),
}

FFP_INPUTS = Path(f"../data/processed/ec/{SITE}_ffp_inputs.csv")
HLS_DIR = Path("../data/raw/hls") / SENSOR
OUT_DIR = Path("../data/processed/vi_tables/ffp")
OUT_DIR.mkdir(parents=True, exist_ok=True)

BANDS = {
    "Landsat":  {"prefix": "L", "blue": "B02", "red": "B04", "nir": "B05", "swir1": "B06"},
    "Sentinel": {"prefix": "S", "blue": "B02", "red": "B04", "nir": "B8A", "swir1": "B11"},
}[SENSOR]

# FFP settings
BL_HEIGHT = 500.0       # boundary layer height (m), not measured at the tower
Z0_DEFAULT = 0.06       # roughness length used when EddyPro gives -999
USTAR_MIN = 0.1         # half hours with weak turbulence are skipped
CONTOUR_PCT = 90

In [ ]:
data = pd.read_csv(FFP_INPUTS)
for c in ["yyyy", "mm", "day"]:
    data[c] = pd.to_numeric(data[c], errors="coerce")
data = data.dropna(subset=["yyyy", "mm", "day"])
data["date"] = pd.to_datetime(dict(year=data["yyyy"].astype(int),
                                   month=data["mm"].astype(int),
                                   day=data["day"].astype(int)))
by_date = data.groupby("date")

scene_folders = sorted(p for p in HLS_DIR.iterdir() if p.is_dir())
scene_dates = [pd.to_datetime(p.name[1:], format="%Y-%m-%d") for p in scene_folders]
print(f"{len(data)} half hours, {len(scene_dates)} {SENSOR} scenes")

## Daily 90% footprint contours

FFP returns the footprint on a grid centred on the tower. The grid is shifted to the tower
position in Web Mercator, the 90% contour is traced, and the result is saved as a polygon in EPSG:4326.

In [ ]:
lat, lon = TOWERS[SITE]
x0, y0 = Transformer.from_crs("EPSG:4326", "EPSG:3857", always_xy=True).transform(lon, lat)

contours = {}
for day in scene_dates:
    if day not in by_date.groups:
        continue

    total = None
    for i, row in by_date.get_group(day).iterrows():
        if row["u_star"] < USTAR_MIN:
            continue
        z0 = row["z0"] if row["z0"] != -999 else Z0_DEFAULT
        try:
            ffp = FFP_climatology(zm=row["zm"], z0=z0, umean=row["U_mean"], h=BL_HEIGHT,
                                  ol=row["L"], sigmav=row["sigma_v"], ustar=row["u_star"],
                                  wind_dir=row["wind_dir"], nx=100, ny=100, verbosity=0)
        except Exception as e:
            print(f"FFP failed for row {i}: {e}")
            continue

        if total is None:
            total = np.zeros(ffp["fclim_2d"].shape)
        if total.shape != ffp["fclim_2d"].shape:
            print(f"grid shape mismatch at row {i}, skipped")
            continue
        total += ffp["fclim_2d"]

    if total is None or total.sum() == 0:
        continue

    # contour level that encloses 90% of the summed footprint
    f = total / total.sum()
    f_sorted = np.sort(f.ravel())[::-1]
    cum_pct = np.cumsum(f_sorted) / f_sorted.sum() * 100
    level = f_sorted[np.searchsorted(cum_pct, CONTOUR_PCT)]

    fig, ax = plt.subplots()
    cs = ax.contour(ffp["x_2d"] + x0, ffp["y_2d"] + y0, f, levels=[level])
    plt.close(fig)

    polys = [{"geometry": Polygon(seg), "percentile": CONTOUR_PCT}
             for seg in cs.allsegs[0] if len(seg) >= 3]
    if polys:
        contours[day] = gpd.GeoDataFrame(polys, crs="EPSG:3857").to_crs("EPSG:4326")

print(f"footprints for {len(contours)} scene dates")

## Indices inside the daily footprint

Same cloud mask as notebook 03 (Fmask bits 1, 3 and 4). A small constant is added to the NDVI and
NDWI denominators to avoid division by zero.

In [ ]:
def clear_sky_values(fmask):
    good = []
    for v in np.unique(fmask):
        v = int(v)
        cloud, shadow, snow = (v >> 1) & 1, (v >> 3) & 1, (v >> 4) & 1
        if not (cloud or shadow or snow):
            good.append(v)
    return good


def find_band(folder, key):
    matches = sorted(f for f in folder.iterdir() if key in f.name)
    if not matches:
        raise FileNotFoundError(f"{key} not found in {folder.name}")
    return matches[0]


def indices_in_footprint(folder, footprint):
    fmask_path = find_band(folder, "Fmask")
    with rasterio.open(fmask_path) as src:
        good = clear_sky_values(src.read(1))
        fp = footprint.to_crs(src.crs) if footprint.crs != src.crs else footprint
        fmask, _ = mask.mask(src, fp.geometry, crop=True, nodata=255)   # HLS Fmask fill value
    clear = np.isin(fmask[0], good)

    b = {}
    for k in ("blue", "red", "nir", "swir1"):
        with rasterio.open(find_band(folder, BANDS[k])) as src:
            arr, _ = mask.mask(src, fp.geometry, crop=True, nodata=-9999)
        arr = arr[0].astype("float64")
        b[k] = np.where(arr == -9999, np.nan, arr) * 0.0001

    blue, red, nir, swir1 = b["blue"], b["red"], b["nir"], b["swir1"]
    with np.errstate(divide="ignore", invalid="ignore"):
        out = {
            "NDVI": (nir - red) / (nir + red + 1e-10),
            "EVI": 2.5 * (nir - red) / (nir + 6.0 * red - 7.5 * blue + 1.0),
            "SAVI": 1.5 * (nir - red) / (nir + red + 0.5),
            "MSAVI": (2 * nir + 1 - np.sqrt((2 * nir + 1) ** 2 - 8 * (nir - red))) / 2,
            "NDWI": (nir - swir1) / (nir + swir1 + 1e-10),
            "SR": nir / red,
        }
    return {k: np.where(clear, v, np.nan) for k, v in out.items()}


records = []
for folder, day in zip(scene_folders, scene_dates):
    if day not in contours:
        continue
    try:
        idx = indices_in_footprint(folder, contours[day])
    except FileNotFoundError as e:
        print(f"skipping {folder.name}: {e}")
        continue
    row = {"Date": day.strftime("%Y-%m-%d")}
    for name, arr in idx.items():
        row[f"{name}_mean"] = np.nanmean(arr)
        row[f"{name}_median"] = np.nanmedian(arr)
    records.append(row)

vi_table = pd.DataFrame(records)
out_path = OUT_DIR / f"{SENSOR}_{SITE}.csv"
vi_table.to_csv(out_path, index=False)
print(f"{len(vi_table)} scenes -> {out_path}")
vi_table.head()